[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/01.%20Clase%201/01Class%20NB.ipynb)

In [ ]:
!pip install -q yfinance pandas numpy matplotlib seaborn scipy scikit-learn

# Clase 1: Análisis de Acciones con Python

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

---

## 1. Descarga de datos históricos con `yfinance`

Usamos el paquete `yfinance` para descargar precios ajustados directamente de Yahoo! Finance.  
Consideramos tres acciones (Alcoa, Apple y Microsoft) y el índice S&P 500.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scipy.stats as stats

# Estilo visual
sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 8)

In [ ]:
tickers     = ["AA", "AAPL", "MSFT", "^GSPC"]
start_date  = "2014-01-01"
end_date    = "2016-12-31"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
print(f"Filas: {len(data)}  |  Columnas: {list(data.columns.get_level_values(0).unique())}")
data.head()

`yfinance` devuelve un `DataFrame` con columnas de dos niveles:  
- **Nivel 0**: campo (`Close`, `Open`, `High`, `Low`, `Volume`)  
- **Nivel 1**: ticker  

Acceder a todos los precios de cierre es directo:

In [ ]:
closes = data["Close"]
closes.head()

In [ ]:
# Precio de cierre de Microsoft
closes["MSFT"].head()

In [ ]:
# Precio de cierre en una fecha específica
closes.loc["2014-01-14"]

In [ ]:
# Todos los campos de un ticker
data.xs("^GSPC", level=1, axis=1).head()

---
## 2. Visualización de precios

### 2.1 Precio ajustado de cierre

In [ ]:
fig, ax = plt.subplots()
closes[["AA", "AAPL", "MSFT"]].plot(ax=ax)
ax.set_title("Precio ajustado de cierre (2014-2016)")
ax.set_ylabel("USD")
plt.tight_layout()

La escala del S&P 500 es muy diferente a la de las acciones individuales,  
por lo que se grafican por separado.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
closes[["AA", "AAPL", "MSFT"]].plot(ax=axes[0], title="Acciones individuales")
axes[0].set_ylabel("USD")
closes["^GSPC"].plot(ax=axes[1], title="S&P 500", color="steelblue")
axes[1].set_ylabel("Puntos")
plt.tight_layout()

### 2.2 Precio y volumen de Microsoft

In [ ]:
msftA = closes["MSFT"]
msftV = data["Volume"]["MSFT"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)
ax1.plot(msftA.index, msftA, label="Precio ajustado")
ax1.set_title("Microsoft: Precio ajustado de cierre 2014-2016")
ax1.set_ylabel("USD")
ax1.legend()

ax2.bar(msftV.index, msftV, width=1, color="steelblue", alpha=0.7)
ax2.set_title("Volumen diario")
ax2.set_ylabel("Volumen")
plt.tight_layout()

### 2.3 Medias móviles

Las **medias móviles** suavizan la serie de precios y ayudan a identificar tendencias.  
Usamos ventanas de 20 y 100 días.

In [ ]:
sma20  = msftA.rolling(20).mean()
sma100 = msftA.rolling(100).mean()

fig, ax = plt.subplots()
ax.plot(msftA,  label="Precio ajustado",       alpha=0.6)
ax.plot(sma20,  label="Media móvil 20 días",   linewidth=2)
ax.plot(sma100, label="Media móvil 100 días",  linewidth=2)
ax.set_title("Microsoft: Precio y medias móviles")
ax.set_ylabel("USD")
ax.legend()
plt.tight_layout()

### 2.4 Bandas de volatilidad (desviación móvil)

In [ ]:
std20  = msftA.rolling(20).std()
std100 = msftA.rolling(100).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, std, window in zip(axes, [std20, std100], [20, 100]):
    ax.plot(msftA, label="Precio ajustado", alpha=0.7, color="steelblue")
    ax.fill_between(msftA.index,
                    msftA - std, msftA + std,
                    alpha=0.25, color="orange",
                    label=f"± 1 std ({window} días)")
    ax.set_ylabel("USD")
    ax.legend()
    ax.set_title(f"Banda de volatilidad — ventana {window} días")
plt.tight_layout()

---
## 3. Rendimientos

### 3.1 Rendimiento simple

Para una serie de precios $\{S_t\}$, el **rendimiento simple** es:
$$
R_t = \frac{S_t - S_{t-1}}{S_{t-1}}
$$

In [ ]:
stocks = closes[["AA", "AAPL", "MSFT"]]
R = stocks.pct_change().dropna()
R.head()

In [ ]:
fig, ax = plt.subplots()
R.plot(ax=ax, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Rendimientos simples diarios (2014-2016)")
ax.set_ylabel("Rendimiento")
plt.tight_layout()

### 3.2 Rendimiento logarítmico

El **rendimiento logarítmico** (continuamente compuesto) se define como:
$$
r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right) = \ln(1 + R_t)
$$
Para rendimientos pequeños, $r_t \approx R_t$.  
El rendimiento log tiene la ventaja de ser **aditivo** en el tiempo.

In [ ]:
r = np.log(stocks / stocks.shift(1)).dropna()
r.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
R.plot(ax=axes[0], alpha=0.7, title="Rendimiento simple")
r.plot(ax=axes[1], alpha=0.7, title="Rendimiento logarítmico")
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Rendimiento")
plt.tight_layout()

### 3.3 Estadísticas descriptivas

In [ ]:
R.describe()

In [ ]:
r.describe()

### 3.4 Volatilidad móvil

La **volatilidad** anualizada se estima como la desviación estándar de los rendimientos multiplicada por $\sqrt{252}$.

In [ ]:
vol_R = R.rolling(20).std() * np.sqrt(252)
vol_r = r.rolling(20).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
vol_R.plot(ax=axes[0], title="Volatilidad anualizada — rendimiento simple (ventana 20 días)")
vol_r.plot(ax=axes[1], title="Volatilidad anualizada — rendimiento log (ventana 20 días)")
for ax in axes:
    ax.set_ylabel("Volatilidad")
plt.tight_layout()

### 3.5 Rendimiento acumulado

El **rendimiento acumulado** muestra el valor de $1 USD invertido al inicio del período.

In [ ]:
cum_R = (1 + R).cumprod()
cum_r = np.exp(r.cumsum())

fig, ax = plt.subplots()
cum_R.plot(ax=ax)
ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Rendimiento acumulado (base = 1 USD, 2014-2016)")
ax.set_ylabel("Valor de la inversión")
plt.tight_layout()

---
## 4. Distribución de los rendimientos

Analizamos si los rendimientos siguen una distribución normal.  
Esta hipótesis es central en muchos modelos financieros (Black-Scholes, CAPM, etc.).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, r.columns):
    r[col].hist(bins=50, ax=ax, edgecolor="white")
    ax.set_title(f"Histograma — {col}")
    ax.set_xlabel("Rendimiento log")
plt.suptitle("Distribución de rendimientos logarítmicos")
plt.tight_layout()

**QQ-Plot**: si los puntos caen sobre la línea roja, la distribución es aproximadamente normal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, r.columns):
    stats.probplot(r[col].dropna(), dist="norm", plot=ax)
    ax.set_title(f"QQ-Plot — {col}")
plt.suptitle("Prueba de normalidad (QQ-Plot)")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots()
r.plot(kind="box", ax=ax)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Boxplot de rendimientos logarítmicos")
ax.set_ylabel("Rendimiento log")
plt.tight_layout()

La **matriz de dispersión** muestra la relación entre los rendimientos de cada par de activos.  
Una correlación positiva alta implica que los activos se mueven juntos (menor diversificación).

In [ ]:
pd.plotting.scatter_matrix(r, diagonal="kde", alpha=0.3, figsize=(9, 9))
plt.suptitle("Matriz de dispersión de rendimientos logarítmicos", y=1.01)
plt.tight_layout()

### Matriz de correlación

In [ ]:
corr = r.corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlación de rendimientos logarítmicos")
plt.tight_layout()